In [28]:
"""
DROUGHT MODE — ML MODEL MODULE
================================
Replaces static Impact_Score with a trained ML pipeline that:
  1. Engineers spatial + environmental features per zone
  2. Trains a Gradient Boosting Regressor (with cross-validated tuning)
  3. Predicts a dynamic Drought_Impact_Score for every zone
  4. Returns a ranked DataFrame ready for the PuLP / Knapsack optimizer

Drop-in replacement: feed this module's output directly into your
existing spatial filter + budget optimizer pipeline.
"""

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, LineString, Point
from shapely.ops import unary_union

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold
from sklearn.inspection import permutation_importance
import warnings

warnings.filterwarnings("ignore")




In [29]:
# ─────────────────────────────────────────────
# SECTION 1 — FEATURE ENGINEERING
# ─────────────────────────────────────────────

def compute_zone_features(gdf_zones: gpd.GeoDataFrame,
                           gdf_water: gpd.GeoDataFrame,
                           buffer_m: int = 500) -> pd.DataFrame:
    """
    Derive a rich feature matrix from spatial geometry alone.
    These features simulate what you'd get from GEE LST / NDVI exports.

    Parameters
    ----------
    gdf_zones   : GeoDataFrame of planting zones (projected to metres, e.g. EPSG:32643)
    gdf_water   : GeoDataFrame of water pipelines (same CRS)
    buffer_m    : drought buffer radius in metres

    Returns
    -------
    DataFrame of engineered features, one row per zone
    """
    feats = {}

    # — Geometry-derived spatial features —
    feats["area_m2"]        = gdf_zones.geometry.area
    feats["perimeter_m"]    = gdf_zones.geometry.length
    feats["compactness"]    = (4 * np.pi * feats["area_m2"]) / (feats["perimeter_m"] ** 2)

    # Centroid coordinates (useful proxy for elevation / urban density gradients)
    feats["centroid_x"]     = gdf_zones.geometry.centroid.x
    feats["centroid_y"]     = gdf_zones.geometry.centroid.y

    # — Water proximity features (key under drought) —
    water_union = unary_union(gdf_water.geometry)

    feats["dist_to_water_m"]   = gdf_zones.geometry.centroid.apply(
        lambda pt: pt.distance(water_union)
    )
    feats["within_buffer"]     = (feats["dist_to_water_m"] <= buffer_m).astype(int)
    feats["water_overlap_pct"] = gdf_zones.geometry.apply(
        lambda z: z.intersection(water_union.buffer(buffer_m)).area / z.area
        if z.area > 0 else 0.0
    )

    # Inverse-distance weighting (zones very close to water score higher)
    feats["idw_water"] = 1.0 / (feats["dist_to_water_m"] + 1.0)

    # — Simulated environmental signals —
    # In production: replace these with real GEE-exported raster stats per zone.
    rng = np.random.default_rng(seed=42)   # fixed seed → reproducible synthetic data

    n = len(gdf_zones)

    # LST (Land Surface Temperature) proxy — hotter zones need more intervention
    feats["lst_mean"]       = rng.uniform(32, 45, n)     # °C, Pune summer range
    feats["lst_std"]        = rng.uniform(0.5, 4.0, n)

    # NDVI proxy — lower NDVI = less vegetation = higher intervention priority
    feats["ndvi_mean"]      = rng.uniform(0.05, 0.45, n)
    feats["ndvi_trend"]     = rng.uniform(-0.05, 0.02, n)  # negative = deteriorating

    # Impervious surface % (urban heat island contribution)
    feats["impervious_pct"] = rng.uniform(30, 90, n)

    # Population density proxy (high density → greater benefit per ₹ spent)
    feats["pop_density"]    = rng.uniform(5000, 50000, n)  # persons/km²

    # Soil moisture under drought
    feats["soil_moisture"]  = rng.uniform(0.05, 0.35, n)  # volumetric water content

    return pd.DataFrame(feats, index=gdf_zones.index)




In [30]:
# ─────────────────────────────────────────────
# SECTION 2 — SYNTHETIC TRAINING LABELS (FOOLPROOF FIX)
# ─────────────────────────────────────────────

def generate_training_labels(X: pd.DataFrame) -> np.ndarray:
    """
    Create physically motivated training labels (0–100 Impact Score).
    Using explicit max() - min() inline to completely avoid Pandas version conflicts.
    """
    score = (
          0.25 * (X["lst_mean"]       - X["lst_mean"].min())       / (X["lst_mean"].max() - X["lst_mean"].min())
        + 0.20 * (1 - (X["ndvi_mean"] - X["ndvi_mean"].min())      / (X["ndvi_mean"].max() - X["ndvi_mean"].min()))
        + 0.20 * X["water_overlap_pct"]
        + 0.15 * (X["pop_density"]    - X["pop_density"].min())    / (X["pop_density"].max() - X["pop_density"].min())
        + 0.10 * (X["impervious_pct"] - X["impervious_pct"].min()) / (X["impervious_pct"].max() - X["impervious_pct"].min())
        + 0.10 * (X["soil_moisture"]  - X["soil_moisture"].min())  / (X["soil_moisture"].max() - X["soil_moisture"].min())
    ) * 100

    # Add calibrated noise so the model generalises rather than memorises
    rng = np.random.default_rng(seed=99)
    score += rng.normal(0, 3, len(score))
    return np.clip(score, 0, 100)

In [31]:
# ─────────────────────────────────────────────
# SECTION 3 — MODEL TRAINING
# ─────────────────────────────────────────────

def build_model(n_augment: int = 300) -> Pipeline:
    """
    Train a Gradient Boosting pipeline on augmented synthetic data.

    Parameters
    ----------
    n_augment : synthetic training samples to generate (≥ 200 recommended)

    Returns
    -------
    Fitted sklearn Pipeline (StandardScaler → GradientBoostingRegressor)
    """
    print(f"  [ML] Generating {n_augment} synthetic training samples...")

    # Augment by perturbing the feature space
    rng = np.random.default_rng(seed=7)
    n   = n_augment

    X_train = pd.DataFrame({
        "area_m2":          rng.uniform(5_000,  200_000, n),
        "perimeter_m":      rng.uniform(300,    2_000,   n),
        "compactness":      rng.uniform(0.2,    1.0,     n),
        "centroid_x":       rng.uniform(8.2e6,  8.25e6,  n),   # UTM easting, Pune belt
        "centroid_y":       rng.uniform(2.04e6, 2.06e6,  n),
        "dist_to_water_m":  rng.uniform(0,      3_000,   n),
        "within_buffer":    rng.integers(0, 2,            n),
        "water_overlap_pct":rng.uniform(0,      1.0,     n),
        "idw_water":        rng.uniform(0.0003, 1.0,     n),
        "lst_mean":         rng.uniform(30,     48,      n),
        "lst_std":          rng.uniform(0.3,    5.0,     n),
        "ndvi_mean":        rng.uniform(0.02,   0.6,     n),
        "ndvi_trend":       rng.uniform(-0.08,  0.05,    n),
        "impervious_pct":   rng.uniform(10,     95,      n),
        "pop_density":      rng.uniform(1_000,  80_000,  n),
        "soil_moisture":    rng.uniform(0.02,   0.5,     n),
    })

    y_train = generate_training_labels(X_train)

    gbr = GradientBoostingRegressor(
        n_estimators      = 400,
        max_depth         = 4,
        learning_rate     = 0.05,
        subsample         = 0.8,
        min_samples_leaf  = 5,
        random_state      = 42,
    )

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("gbr",    gbr),
    ])

    # Cross-validation to verify generalisation
    cv     = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=cv,
                              scoring="r2", n_jobs=-1)
    print(f"  [ML] 5-fold CV R² scores: {np.round(scores, 3)}")
    print(f"  [ML] Mean R² = {scores.mean():.3f} ± {scores.std():.3f}")

    model.fit(X_train, y_train)
    print("  [ML] Model trained ✓")
    return model




In [32]:
# ─────────────────────────────────────────────
# SECTION 4 — INFERENCE + RANKING
# ─────────────────────────────────────────────

def predict_drought_impact(gdf_zones: gpd.GeoDataFrame,
                            gdf_water: gpd.GeoDataFrame,
                            model:     Pipeline,
                            buffer_m:  int = 500) -> gpd.GeoDataFrame:
    """
    Predict a Drought_Impact_Score for every zone and attach it to the GDF.

    Parameters
    ----------
    gdf_zones  : projected GeoDataFrame of zones
    gdf_water  : projected GeoDataFrame of water lines
    model      : fitted sklearn Pipeline from build_model()
    buffer_m   : drought buffer radius

    Returns
    -------
    GeoDataFrame with new columns:
        Drought_Impact_Score  — ML-predicted 0–100 urgency score
        Drought_Priority_Rank — 1 = highest priority
    """
    X_infer = compute_zone_features(gdf_zones, gdf_water, buffer_m)
    preds   = model.predict(X_infer)

    gdf_out = gdf_zones.copy()
    gdf_out["Drought_Impact_Score"]  = np.clip(preds, 0, 100).round(2)
    gdf_out["Drought_Priority_Rank"] = (
        gdf_out["Drought_Impact_Score"]
        .rank(ascending=False, method="min")
        .astype(int)
    )
    return gdf_out




In [33]:
# ─────────────────────────────────────────────
# SECTION 5 — FEATURE IMPORTANCE REPORT
# ─────────────────────────────────────────────

def feature_importance_report(model: Pipeline, feature_names: list) -> pd.DataFrame:
    """
    Extract and display feature importances from the trained GBR.
    """
    importances = model.named_steps["gbr"].feature_importances_
    df_imp = pd.DataFrame({
        "Feature":   feature_names,
        "Importance": importances,
    }).sort_values("Importance", ascending=False).reset_index(drop=True)

    print("\n  [ML] Feature Importances (GBR):")
    print(df_imp.to_string(index=False))
    return df_imp




In [34]:
# ─────────────────────────────────────────────
# SECTION 6 — MAIN DEMO (mirrors your script's structure)
# ─────────────────────────────────────────────

if __name__ == "__main__":

    print("=" * 60)
    print("  DROUGHT MODE ACTIVATED — ML-POWERED IMPACT PREDICTION")
    print("=" * 60)

    # ── Recreate your existing GeoDataFrames ──────────────────────
    neighborhoods_data = {
        "Zone_ID":      ["Kothrud_A", "Viman_Nagar_B", "Hinjewadi_C",
                         "Shivajinagar_D", "Baner_E"],
        "Cost_INR":     [100_000, 150_000, 120_000, 90_000, 110_000],
        "Geometry": [
            Polygon([(73.80,18.52),(73.81,18.52),(73.81,18.53),(73.80,18.53)]),
            Polygon([(73.91,18.56),(73.92,18.56),(73.92,18.57),(73.91,18.57)]),
            Polygon([(73.73,18.59),(73.74,18.59),(73.74,18.60),(73.73,18.60)]),
            Polygon([(73.84,18.53),(73.85,18.53),(73.85,18.54),(73.84,18.54)]),
            Polygon([(73.77,18.55),(73.78,18.55),(73.78,18.56),(73.77,18.56)]),
        ],
    }
    gdf_zones = gpd.GeoDataFrame(
        neighborhoods_data, geometry="Geometry", crs="EPSG:4326"
    ).to_crs(epsg=32643)

    water_lines_data = {
        "Pipeline_Name": ["Main_Line_1", "Main_Line_2"],
        "Geometry": [
            LineString([(73.79,18.51),(73.82,18.54)]),
            LineString([(73.72,18.58),(73.75,18.61)]),
        ],
    }
    gdf_water = gpd.GeoDataFrame(
        water_lines_data, geometry="Geometry", crs="EPSG:4326"
    ).to_crs(epsg=32643)

    BUFFER_M = 500

    # ── Step 1: Train the ML model ────────────────────────────────
    print("\n[STEP 1] Training ML model...")
    model = build_model(n_augment=400)

    # ── Step 2: Predict impact scores ────────────────────────────
    print("\n[STEP 2] Predicting Drought Impact Scores...")
    gdf_scored = predict_drought_impact(gdf_zones, gdf_water, model, BUFFER_M)

    # ── Step 3: Apply your existing spatial water filter ──────────
    print("\n[STEP 3] Applying spatial drought water filter...")
    gdf_water_buf = gdf_water.copy()
    gdf_water_buf["geometry"] = gdf_water.geometry.buffer(BUFFER_M)
    water_union_buf = unary_union(gdf_water_buf.geometry)

    gdf_scored["Water_Available"] = gdf_scored.geometry.intersects(water_union_buf)

    # ── Step 4: Filter & rank viable zones ───────────────────────
    viable = (
        gdf_scored[gdf_scored["Water_Available"]]
        .sort_values("Drought_Priority_Rank")
        .reset_index(drop=True)
    )

    # ── Step 5: Results ───────────────────────────────────────────
    print("\n" + "=" * 60)
    print("  DROUGHT FILTER COMPLETE — ML-RANKED INTERVENTION LIST")
    print("=" * 60)

    display_cols = ["Zone_ID", "Drought_Impact_Score", "Drought_Priority_Rank",
                    "Water_Available", "Cost_INR"]
    print(gdf_scored[display_cols].to_string(index=False))

    print("\n  Viable zones (water-accessible), ranked for optimizer:")
    print(viable[["Zone_ID", "Drought_Impact_Score",
                  "Drought_Priority_Rank", "Cost_INR"]].to_string(index=False))

    print("\n  Zone IDs ready for PuLP / Knapsack solver:")
    print(viable["Zone_ID"].tolist())

    # ── Step 6: Feature importance ────────────────────────────────
    feature_names = compute_zone_features(gdf_zones, gdf_water, BUFFER_M).columns.tolist()
    feature_importance_report(model, feature_names)

    print("\n  [DONE] ML model module complete.")
    print("  Plug `viable` DataFrame directly into your PuLP optimizer.\n")

  DROUGHT MODE ACTIVATED — ML-POWERED IMPACT PREDICTION

[STEP 1] Training ML model...
  [ML] Generating 400 synthetic training samples...
  [ML] 5-fold CV R² scores: [0.915 0.888 0.899 0.87  0.857]
  [ML] Mean R² = 0.886 ± 0.020
  [ML] Model trained ✓

[STEP 2] Predicting Drought Impact Scores...

[STEP 3] Applying spatial drought water filter...

  DROUGHT FILTER COMPLETE — ML-RANKED INTERVENTION LIST
       Zone_ID  Drought_Impact_Score  Drought_Priority_Rank  Water_Available  Cost_INR
     Kothrud_A                 60.29                      1             True    100000
 Viman_Nagar_B                 35.43                      5            False    150000
   Hinjewadi_C                 57.68                      2             True    120000
Shivajinagar_D                 37.06                      4            False     90000
       Baner_E                 38.14                      3            False    110000

  Viable zones (water-accessible), ranked for optimizer:
    Zone_ID  